# 03 — Inference Strategies + ML Training with Lance

**Purpose:** Demonstrate three progressively scalable approaches to scene classification (weather, scene type, time of day) using the Lance `frames` dataset. The right choice depends on your scale and cost constraints — this notebook makes those trade-offs explicit.

---

## Why `ray.data.read_lance` instead of `ray.data.read_databricks_tables`?

This notebook uses `ray.data.read_lance` as the data loading primitive. The alternatives (`ray.data.read_databricks_tables`, `ray.data.read_parquet` against a Delta table) introduce overhead that compounds at training scale.

### 1. Random access without shuffle overhead

ML training requires continuous data shuffling to prevent overfitting. Parquet forces Ray Data to read entire row groups (~128MB each) to retrieve a handful of samples — if Ray needs rows 1, 50,000, and 2,000,000, it reads three full row groups. Lance uses fragment-level addressing for O(1) point lookups. [LanceDB benchmarks](https://lancedb.github.io/lance/) place random access at 100–1000x faster than Parquet depending on access pattern. At 10M+ frames, this is the difference between data loading being a bottleneck or not.

### 2. Near-zero deserialization overhead (Arrow-native)

Ray Data's in-memory engine represents all data as Apache Arrow blocks. Reading from Parquet/Delta requires: read from storage → deserialize Parquet bytes → convert to Arrow. That deserialization step burns CPU cycles on every batch — cycles that could go to preprocessing and augmentation. Lance stores data in Arrow IPC format natively, so Ray Data reads directly into Arrow blocks with no conversion step.

### 3. Fragment-parallel reads map to Ray's actor model

A Lance dataset is composed of independent fragments. Each Ray worker actor reads its assigned fragment(s) without coordinating with other workers — embarrassingly parallel by design. Separately, Lance isolates large binary payloads (JPEG image bytes) in a blob layout distinct from the metadata columns. A Ray worker filtering on `video_id` or reading only the `embedding` column never touches image bytes — the heavy payload doesn't tax metadata scans.

---

## Why not just use a VLM for everything?

For well-understood visual categories like weather or scene type, modern vision-language models work zero-shot — no training required. The case for a trained classifier is almost entirely about **cost and throughput at scale**. At 10M frames, VLM API inference runs to tens of thousands of dollars and takes weeks. A trained classifier on stored CLIP embeddings costs pennies and takes hours.

This notebook shows all three approaches so you can choose based on your dataset size and latency requirements.

---

## Stage 1 — VLM Structured Output (LLaVA via Ray)

Run a vision-language model (LLaVA, served locally on Ray GPU workers) against a sample of frames. Prompt the model to return structured JSON with weather, scene type, and time-of-day fields. This approach produces the richest, most flexible output and requires no labeled training data — but is too expensive and slow to run against millions of frames.

**Use when:** Dataset is small (<100K frames), labels are ambiguous or open-ended, or you need to generate pseudo-labels for an unlabeled dataset.

## Stage 2 — CLIP Zero-Shot (uses stored embeddings, no training)

Use the CLIP embeddings already stored in Lance to classify frames via text-image cosine similarity. Encode each label as a natural language prompt (e.g., `"a dashcam photo taken in rainy weather"`) and find the closest match to each frame's stored embedding. No model training, no API calls — inference runs directly against the Lance dataset in milliseconds.

Benchmarks on BDD100K place CLIP zero-shot at ~70–75% accuracy for weather classification. For many production use cases this is sufficient, and the cost is effectively zero.

**Use when:** Label space is well-defined, accuracy requirements are moderate, and you want a fast production baseline with no training overhead.

## Stage 3 — CLIP Embeddings → Trained MLP Classifier

Train a lightweight MLP on top of frozen CLIP embeddings using BDD100K ground-truth labels (weather × scene × time-of-day). Load training batches via `ray.data.read_lance` with column projection (`embedding` + labels only — no image bytes loaded at training time since CLIP embeddings are sufficient). Three classification heads share the same embedding input.

Includes a data loading benchmark: `ray.data.read_lance` vs `ray.data.read_parquet` against an equivalent Delta table at the same row count. Reports samples/second for both — the gap widens past 10M rows where Delta's shuffle forces full row-group reads on every batch.

Log loss, per-head accuracy, and data loading throughput to MLflow. Pin the Lance dataset version as a run parameter for full reproducibility.

**Use when:** You need domain-adapted accuracy, deterministic inference, or throughput at scale (10M+ frames). Fine-tuning adapts to your specific camera hardware and data distribution in ways zero-shot cannot.

---

**Inputs:** Lance `frames` dataset v2 at `/Volumes/{catalog}/{schema}/{volume}/frames/` (with CLIP embeddings); BDD100K video-level labels joined to frames via `video_id`

**Outputs:** Stage 1 — structured VLM predictions on a sample; Stage 2 — zero-shot accuracy report; Stage 3 — trained classifier logged to MLflow + `ray.data.read_lance` vs `ray.data.read_parquet` throughput benchmark